<a href="https://colab.research.google.com/github/psullivan2213/aggregate-CC-transactions/blob/main/Aggregate_CC_Transactions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Import

In [ ]:
import pandas as pd
import os

In [ ]:
#define function to rename csv file
def rename_csv(uploaded, rename):
    for name, content in uploaded.items():
        if name != rename:  # Using "!="
            # Save the uploaded content to a file with original name
            with open(name, 'wb') as f:
                f.write(content)
            # Rename the file to 'DiscoverTransactions.csv'
            os.rename(name, rename)
            print(f"Renamed '{name}' to '{rename}'")
            break  # Exit the loop after renaming

##Upload and Name Discover Transactions

In [ ]:
#upload DISCOVER CSV here
from google.colab import files
uploaded1 = files.upload()

In [ ]:
#call function to rename file to "DiscoverTransactions"
rename_csv(uploaded1, 'DiscoverTransactions.csv')

In [ ]:
#create disdf
disdf = pd.read_csv('DiscoverTransactions.csv')
disdf.head()

##Upload and Name Cap One Transactions

In [ ]:
#upload CAPITAL ONE CSV here
uploaded2 = files.upload()

In [ ]:
#call function to rename file to "CapitalOneTransactions"
rename_csv(uploaded2, 'CapitalOneTransactions.csv')

In [ ]:
#create capdf
capdf = pd.read_csv('CapitalOneTransactions.csv')
capdf.head()

##Upload and Name US Bank Transactions

In [ ]:
#upload US BANK CSV
uploaded3 = files.upload()

In [ ]:
#call function to rename file to "USBankTransactions"
rename_csv(uploaded3, 'USBankTransactions.csv')

In [ ]:
#create usbankdf
usbankdf = pd.read_csv('USBankTransactions.csv')
usbankdf.head()

##Create Master Dataframe by concatenating the CSVs

In [ ]:
#create alldf to encompass all transactions from each csv
alldf = pd.concat([disdf, capdf, usbankdf])
alldf

In [ ]:
#delete row if description is "CAPITAL ONE MOBILE PYMT"
alldf = alldf[alldf['Description'] != 'CAPITAL ONE MOBILE PYMT']
alldf

In [ ]:
#reorder index
alldf.reset_index(drop=True, inplace=True)
alldf

In [ ]:
#merge trans. date, date, and transaction date columns
if 'Trans. Date' and 'Date' in alldf.columns:
    alldf['Transaction Date'] = alldf['Transaction Date'].fillna(alldf['Trans. Date'])
    alldf['Transaction Date'] = alldf['Transaction Date'].fillna(alldf['Date'])
    alldf.drop(columns=['Trans. Date'], inplace=True)
    alldf.drop(columns=['Date'], inplace=True)
    print("Columns merged successfully.")
    #standardize date format
    alldf['Transaction Date'] = pd.to_datetime(alldf['Transaction Date'], format='mixed').dt.strftime('%m/%d/%Y')
    print("Date format standardized.")
else:
    print("Column 'Trans. Date' and column 'Date' not found in the DataFrame.")
alldf.head()

In [ ]:
#merge Post Date and Posted Date columns
if 'Post Date' in alldf.columns and 'Posted Date' in alldf.columns:
    alldf['Posted Date'] = alldf['Posted Date'].fillna(alldf['Post Date'])
    alldf.drop(columns=['Post Date'], inplace=True)
    print("Columns merged successfully.")
    #standardize date format
    alldf['Posted Date'] = pd.to_datetime(alldf['Posted Date'], format='mixed').dt.strftime('%m/%d/%Y')
    print("Date format standardized.")
else:
    print("Column 'Post Date' not found in the DataFrame.")
alldf.head()

In [ ]:
#merge description and names columns
if 'Description' and 'Name' in alldf.columns:
    alldf['Description'] = alldf['Description'].fillna(alldf['Name'])
    alldf.drop(columns=['Name'], inplace=True)
    print("Columns merged successfully.")
else:
    print("Column 'Name' not found in the DataFrame.")
alldf.head()

In [ ]:
#drop Card No. column
if 'Card No.' in alldf.columns:
    alldf.drop(columns=['Card No.'], inplace=True)
    print("Column 'Card No.' dropped successfully.")
else:
    print("Column 'Card No.' not found in the DataFrame.")

In [ ]:
#fix usbank amounts to positive amounts for debit transactions and negative amounts for credit transactions
for index, row in alldf.iterrows():  # Iterate through rows
    if row['Transaction'] == 'DEBIT':
        alldf.loc[index, 'Amount'] = abs(row['Amount'])  # Use absolute value for DEBIT transactions
    elif row['Transaction'] == 'CREDIT':
        alldf.loc[index, 'Amount'] = -row['Amount']  # Negate 'Amount' for CREDIT transactions
    else:
        print(f"NaN Transaction for row {index}")

# Change data under 'Credit' column to negative amount if it exists
if 'Credit' in alldf.columns:
    alldf['Credit'] = -alldf['Credit']  # Negate 'Credit' column
    print("Column 'Credit' updated to negatives successfully.")
else:
    print("Unsuccessful: 'Credit' column not found")
#move debit and credit data under amount column
if 'Debit' and 'Credit' in alldf.columns:
    alldf['Amount'] = alldf['Amount'].fillna(alldf['Debit'])
    alldf['Amount'] = alldf['Amount'].fillna(alldf['Credit'])
    alldf.drop(columns=['Debit'], inplace=True)
    alldf.drop(columns=['Credit'], inplace=True)
    print("Columns merged successfully.")
else:
    print("Columns 'Debit' and 'Credit' not found in the DataFrame.")
alldf

In [ ]:
#drop transaction column
if 'Transaction' in alldf.columns:
    alldf.drop(columns=['Transaction'], inplace=True)
    print("Column 'Transaction' dropped successfully.")
else:
    print("Column 'Transaction' not found in the DataFrame.")
#drop memo column
if 'Memo' in alldf.columns:
    alldf.drop(columns=['Memo'], inplace=True)
    print("Column 'Memo' dropped successfully.")
else:
    print("Column 'Memo' not found in the DataFrame.")

In [ ]:
#export alldf as excel
alldf.to_excel('CCTransactions.xlsx', index=False)